# pyBIS Workshop Level 0 — From GUI to Automation
**120 minutes | Beginner friendly | BAM Lab Notebook example**

**Three successes:** (1) retrieve my openBIS data; (2) modify metadata and verify it in the GUI; (3) automate a repetitive task.

This notebook is adapted from the supplied `lab-notebook-pybis.ipynb` and the [BAM Lab Notebook tutorial](https://bamresearch.github.io/openBIS-data-store-tutorials/lab-notebook/). It deliberately **does not** reproduce the full tutorial. It uses a project called `Material_Synthesis`, collections named **Synthesis**, **Parameter 1**, and **Parameter 2**, and a few `EXPERIMENTAL_STEP` objects.

**Important:** Work only in your assigned training Space. The notebook creates a dedicated `dummy_sample` object for editing; do not modify or delete original GUI tutorial objects. No terminal commands are needed if the instructor has prepared the environment.

### Code to test if environment and dependencies have been installed

In [ ]:
import sys
from pathlib import Path

expected_env_name = "pybis2"#will be changed to pybis for users
active_prefix = Path(sys.prefix)

is_conda_env = (active_prefix / "conda-meta").is_dir()
is_expected_env = active_prefix.name.casefold() == expected_env_name.casefold()
is_python_314 = sys.version_info[:2] == (3, 14)

print("Python executable:", sys.executable)
print("Environment:", active_prefix)
print("Conda environment:", is_conda_env)
print("Expected environment (pybis):", is_expected_env)
print("Python version:", sys.version.split()[0])

if not (is_conda_env and is_expected_env and is_python_314):
    raise RuntimeError("Select the pybis Conda environment with Python 3.14, then rerun this cell.")

print("Environment check passed.")

In [ ]:
from importlib.metadata import PackageNotFoundError, version

print("Python executable:", sys.executable)

for package in ("pybis", "ipykernel"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError as error:
        raise RuntimeError(f"{package} is not installed in this environment.") from error

print("Both packages are installed and import successfully.")

### Important Libraries

In [ ]:
#commented libraries are there in case some users experience issues

# # %pip install pybis
# %pip install ipykernel
%pip install zipfile
# # %pip install jupyterlab

## Part A — Connect and prepare (guided)
### 1. Connect to openBIS
1. Open the BAM Data Store playground in your browser and log in.
2. In **Tools → User Profile**, copy your openBIS session token (as described in the source notebook).
3. Run the next cell with **Shift + Enter**. Paste the token into the hidden prompt. **Never paste a token into a saved code cell or share it.**

In [ ]:
from getpass import getpass
import pandas as pd
from pybis import Openbis
from helper import *

o = Openbis("https://playground.datastore.bam.de/")
o.set_token(getpass("Paste your openBIS session token (hidden): "))
print("pyBIS connection configured. Next, select your Space.")

### 2. Select your own Space
Uncomment the first line to see accessible Spaces. Replace the placeholder with your assigned Space code.

In [ ]:
# display(o.get_spaces())
myspace = o.get_space('your_space_code_here')

### 3. Separating Paths
**Path A:** You completed the GUI tutorial. The notebook looks for the existing `Material_Synthesis` project and its named collections.

**Path B:** You did not complete the GUI tutorial. The notebook creates **only missing** minimal structures: a project, three collections and a few demonstration experimental steps.

Run the same cells for either path. **No need to decide manually.** The checks are based on actual objects.

In [ ]:
PROJECT_CODE = "Material_Synthesis"

sample_codes = {
    'Macrostructure Characterization': 'Parameter 1',
    'Macrostructure Characterization TS': 'Parameter 1',
    'Microstructure Characterization': 'Parameter 2',
    'Material Synthesis': 'Synthesis'
}
collection_codes = ["Parameter 1", "Parameter 2", "Synthesis"]
project_missing_flag = False
missing_collections = []
missing_samples = []
path = 'B'#default value


try:#try except for catching no project error
    project = myspace.get_project(project_code=PROJECT_CODE)
except Exception:
    print('Project Material Synthesis Missing')
    project_missing_flag = True
    missing_collections = collection_codes.copy()

if not project_missing_flag:#if project exists
    project_collections = project.get_collections()
    collections_by_name = {
        collection.props.get('$name'): collection
        for collection in project_collections
    }
    missing_collections = [
        name for name in collection_codes if name not in collections_by_name
    ]
    for template_name, collection_name in sample_codes.items():
        collection = collections_by_name.get(collection_name)
        if collection is None:
            continue
        existing_sample_names = {
            sample.props.get('$name') for sample in collection.get_samples()
        }
        if template_name not in existing_sample_names:
            missing_samples.append((template_name, collection_name))

if not project_missing_flag and not missing_collections and not missing_samples:
    print('Project, collections, and expected samples exist')
    path = 'A'

print('Missing collections:', missing_collections)
print('Missing samples:', missing_samples)
print('Path =', path)

In [ ]:
# If Path B, create only the entities that are missing.
if path == 'B':
    if project_missing_flag:
        project = myspace.new_project(
            code=PROJECT_CODE,
            description='This tutorial project demonstrates a simplified workflow for the synthesis and characterization of a material within openBIS. The experimental descriptions, metadata, and datasets were generated using AI and represent synthetic example data created solely for training and demonstration purposes. Although the content has been reviewed for consistency and usability within the tutorial, it has not been validated for scientific accuracy and should not be considered scientifically reliable or reproducible.'
        )
        project.save()
        collections_to_create = collection_codes
    else:
        collections_to_create = missing_collections

    for name in collections_to_create:
        collection = o.new_collection(
            code=auto_generated_code(project),
            type='Collection',
            project=project.identifier,
            props={'$name': name}
        )
        collection.save()

    collections_by_name = {
        collection.props.get('$name'): collection
        for collection in project.get_collections()
    }

    for template_name, collection_name in sample_codes.items():
        collection = collections_by_name[collection_name]
        existing_sample_names = {
            sample.props.get('$name') for sample in collection.get_samples()
        }
        if template_name in existing_sample_names:
            continue

        if template_name == 'Macrostructure Characterization TS':
            sample_props = templates['Macrostructure Characterization'].copy()
        else:
            sample_props = templates[template_name].copy()
        sample_props['$name'] = template_name

        template_sample = o.new_sample(
            type='EXPERIMENTAL_STEP',
            collection=collection,
            props=sample_props
        )
        template_sample.save()

**GUI checkpoint:** Open the Material Synthesis project. Can you find **Synthesis**, **Parameter 1**, **Parameter 2** and a total of four Experimental Steps/Samples called:
'Macrostructure Characterization', 'Macrostructure Characterization TS', 'Microstructure Characterization', and 'Material Synthesis'? If yes, you are ready for the hands-on exercises.

---
## Part B — Hands-on challenges
### Challenge 1 — Retrieve your data (30 minutes)
**First success:** *I can execute Python code and retrieve my openBIS data.*

**Task:** Retrieve the project and print its collections. Then find one experimental step/sample by its identifier. You only need to change one line in the challenge cell.

In [ ]:
print("Project:", project.identifier)

#We use the Parameter 2 collection and print the identifier for the experimental step called Microstructure Characterization

collection = None
samples = None
for collection in project.get_collections():
    if collection.props.all()["$name"] == 'Parameter 2':#We select Parameter 2
        samples = collection.get_samples()
        break


print('Collection: ',collection.props.all()["$name"])
print('Sample: ',samples[0].props.all()['$name'])
print('Sample Identifier: ',samples[0].identifier)

In [ ]:
# YOUR TURN: replace name of the collection to Synthesis in order to find the identifier for the Experimental Step/Sample called Material Synthesis


samples = None
for collection in project.get_collections():
    if collection.props.all()["$name"] == 'Parameter 2':# <-- Change the name from Parameter 2 to Synthesis
        samples = collection.get_samples()
        break

print('Sample: ',samples[0].props.all()['$name'])

"Synthesis", 
print('Sample Identifier: ',samples[0].identifier)

**Check:** Does the printed object match the experimental step you can see in the GUI?


### Challenge 2 — Explore and filter (30 minutes)
**Task:** Build a table from the objects in our two collections, inspect it and filter to one collection. 

In [ ]:
rows = []
for collection_name, collection in [("Synthesis", next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Synthesis')), 
                                    ("Parameter 1", next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Parameter 1'))]:
    
    for obj in collection.get_samples():
        rows.append({
            "collection": collection_name,
            "identifier": obj.identifier,
            "name": obj.props.get("$name"),
            "type": obj.type,
        })

objects_table = pd.DataFrame(rows, columns=["collection", "identifier", "name", "type"])
display(objects_table)
print("Total objects:", len(objects_table))

In [ ]:
# YOUR TURN: Explore a different collection. Change the value of the variable 'collection_to_find' to 'Synthesis' and run the cell.


collection_to_find = "Parameter 1"  # <-- change this
filtered = objects_table[objects_table["collection"] == collection_to_find]
display(filtered)

# How many Objects are registered within this collection?
print("Matching objects:", len(filtered))

**Hint:** The text must match a collection name exactly. Try changing `collection_to_find` and running the cell again.

### Challenge 3 — Modify metadata (30 minutes)
**Second success:** *I can change an existing object's metadata and see the result in the GUI.*

**Task:** Edit **only** the `dummy_sample` Experimental Step. Preview the change, then explicitly choose to save it. Verify in the GUI.

In [ ]:
#We register a dummy sample for demo purposes in the collection Synthesis, then we change the experimental description

col = None
for collection in project.get_collections():
    if collection.props.all()["$name"] == 'Synthesis':
        col = collection

sample = o.new_sample(
        type       = 'EXPERIMENTAL_STEP',
        collection =  col,
        props      =  {'$name': 'dummy_sample'}
    )
sample.save()

PROPERTY = "experimental_step.experimental_description"
print("Current description:", sample.props.get(PROPERTY))

In [ ]:
new_description = "I updated this training record using pyBIS!"  # <-- personalize this
print("Proposed description:", new_description)

In [ ]:
sample.props[PROPERTY] = new_description    
sample.save()

In [ ]:
# Verify from a fresh read of the server.
verified = o.get_sample(sample.identifier)
print("Description currently in openBIS:", verified.props.get(PROPERTY))

**GUI checkpoint:** Refresh the practice object in the Lab Notebook. Do you see your updated description?

### Challenge 4 — Automate and export (30 minutes)
**Third success:** *I can automate a task that would otherwise require repetitive GUI operations.*

**Task:** Retrieve all experimental steps in the two selected collections and export a CSV report. This uses a `for` loop; you do not need to write the loop from scratch. Only change the filename or selected collection(s).

In [ ]:
# Choose the collections to include in the report.


selected_collections = [next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Synthesis'), 
                        next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Parameter 1')]

report_rows = []
for collection in selected_collections:
    for obj in collection.get_samples():
        report_rows.append({
            "collection": collection.props.get("$name"),
            "identifier": obj.identifier,
            "name": obj.props.get("$name"),
            "description": obj.props.get("experimental_step.experimental_description"),
        })

report = pd.DataFrame(
    report_rows, columns=["collection", "identifier", "name", "description"]
)

display(report)

filename = "my_openbis_experimental_steps.csv"  # <-- customize
report.to_csv(filename, index=False)

print(f"Exported {len(report)} records to {filename}")

**Check:** Open the generated CSV. Does it contain the names you saw in the GUI?

**Optional:** Edit any cells in the CSV file to see the changes reflected in the next code cell.

In [ ]:
# We will now read the generated CSV file called 'my_openbis_experimental_steps.csv' or the name you customized it to.

table = pd.read_csv(filename) # <-- the name of the file is stored in the filename variable so you do not need to change anything here even if you customized 
                                    # the name of the file in the previous cell

display(table)# <-- This will display the contents of the excel file

### Challenge 5 - Uploading several datasets to one Experimental Step / Sample (30 minutes)

**Task:** Automate data upload (several formats) to the Experimental Step / Sample "dummy_sample"


In [ ]:
# This cell extracts the zipfile into a folder for allowing upload of multiple files at once

import zipfile
import os

zip_path = "datasets/Parameter1_Macroscopic.zip"
extract_dir = "datasets/Parameter1_Macroscopic"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print(f"Extracted {zip_path} to {extract_dir}")

In [ ]:
# We store the collection 'Synthesis' and the Experimental Step 'dummy_sample' into variables

synthesis_collection = next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Synthesis')

dummy_sample = next(sample for index, sample in enumerate(synthesis_collection.get_samples()) if sample.props.all()['$name'] == 'dummy_sample')


ds_new = o.new_dataset(
    type       = 'RAW_DATA',#the code from the table above if run
    sample     = dummy_sample.identifier,
    file    = 'datasets/Parameter1_Macroscopic',#this is the dataset we extracted in the previous cell
    props      = {'$name': 'Parameter 1 raw' ,'notes' : 'Synthetic example images for demo purpose.' }#We give it a name and some comments
)
ds_new.save()

GUI Checkpoint: Check the dummy_sample Experimental Step, Do you see the RAW_DATA we just added?

In [ ]:
# We now add two more types of datasets

ds_new = o.new_dataset(
    type       = 'ELN_PREVIEW',
    sample     = dummy_sample.identifier,
    file       = 'datasets/Synthetic-image1.png',
    props      = {'$name': 'Synthetic image1' ,'notes' : 'This is an artificial example created for demonstration testing.' }
)
ds_new.save()

ds_new = o.new_dataset(
    type       = 'ELN_PREVIEW',# <-- change this to ATTACHMENT
    sample     = dummy_sample.identifier,
    file       = 'datasets/README-Synthetic-image1.txt',#note that the variable name is now file instead of zipfile because its a normal file and not a zip file
    props      = {'$name': 'README Synthetic image1' ,'notes' : 'This is a README file to explain synthetic image1.' }
)
ds_new.save()


GUI Checkpoint: Check the dummy_sample Experimental Step, How many Datsets do you see?



---
## Part C — Wrap-up (5 minutes)
- **Retrieve:** Which pyBIS command gets an existing experimental step?
- **Modify:** What does `.save()` do? Why should you verify in the GUI?
- **Automate:** Which manual GUI task could you replace with a loop and a CSV export?

**Takeaway:** You do not have to memorize Python. You can start with working code, change a few values, and automate a useful openBIS task.

### Instructor notes / optional hints
- **Path A vs B:** Cells 3–5 reuse existing objects by project code and displayed `$name`; only missing items are created. A participant who completed the GUI tutorial may still get a dedicated practice object.
- **Data isolation:** Assign each participant their own Space or a separate project; do not have participants edit the same practice object.
- **Compatibility:** Test the code against the installed pyBIS version and training instance. In particular verify `get_collections()`, `get_samples()`, the collection type code `Collection`, and the description property.
- **Potential name collision:** If participants already created a project with a different code, update `PROJECT_CODE` before the workshop. If a collection has the same code but a different name, resolve manually rather than guessing.
- **Authentication:** Tokens must be entered via a hidden prompt and never stored in notebook outputs. Clear notebook outputs before distribution.
- **Excluded on purpose:** `helper.py`, ZIP downloads, dataset uploads, deletion, additional collections and advanced dataset search from the source notebook.
- **Fallback:** If connection or object creation fails, have a prepopulated individual project ready so participants can start at Challenge 1.
- **Solutions:** Challenge 1 → `chosen_id = macro_step.identifier`; Challenge 2 → `collection_to_find = "Parameter 1"`; Challenge 3 → set `SAVE_CHANGE = True` after preview; Challenge 4 → `selected_collections = [synthesis]`.